# GSM8K baselines on Colab

Greedy pass@1 on the GSM8K test set (1,319 problems) for every model in `common/models.py`.
Results go to Google Drive, and finished runs are skipped, so after a disconnect just run all cells again.

**Before running**
1. *Runtime > Change runtime type*: pick a GPU. T4 (free) works but is slow (no bf16, so float32); L4 or A100 is much faster.
2. *Secrets* (key icon, left sidebar): add `GITHUB_TOKEN`, a fine-grained GitHub token with read access to this repo's contents (the repo is private). Turn on notebook access.
   Optional: `HF_TOKEN` for faster Hugging Face downloads.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Setup: Drive, code, packages, data

In [ ]:
import os
from google.colab import drive, userdata

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/implicit-adaptation"
RESULTS = f"{DRIVE}/results"   # full runs
SMOKE = f"{DRIVE}/smoke"       # 50-question checks
os.makedirs(RESULTS, exist_ok=True)

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("no HF_TOKEN secret; downloads are unauthenticated (fine, just slower)")

In [ ]:
REPO = "/content/implicit-adaptation"
token = userdata.get("GITHUB_TOKEN")
if not os.path.exists(REPO):
    !git clone -q https://{token}@github.com/ignoreandfly/implicit-adaptation.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!git log --oneline -1

In [ ]:
!pip install -q -r requirements.txt
import torch, transformers
print(torch.__version__, transformers.__version__)

In [ ]:
if not os.path.exists("data/gsm8k/test.jsonl"):
    !python -m tasks.gsm8k --out data/gsm8k
!wc -l data/gsm8k/*.jsonl

## 2. Run list

Base models get the plain `Question:/Answer:` prompt (zero-shot and 8-shot); post-trained models get their chat template (zero-shot).
Batch sizes are set from the GPU; lower them if you hit out-of-memory.

In [ ]:
import json, glob

RUNS = [  # (registry name, few-shot exemplars)
    ("olmo2-1b-base", 0), ("olmo2-1b-base", 8),
    ("olmo2-1b-sft", 0), ("olmo2-1b-dpo", 0), ("olmo2-1b-instruct", 0),
    ("qwen2.5-0.5b-base", 0), ("qwen2.5-0.5b-base", 8),
    ("qwen2.5-0.5b-instruct", 0),
]

bf16_gpu = torch.cuda.get_device_capability()[0] >= 8   # L4 / A100
BS_ZERO, BS_FEW = (64, 16) if bf16_gpu else (32, 8)     # 8-shot prompts are ~10x longer

def run(model, shots, root, limit=None):
    out = f"{root}/{model}/gsm8k/{shots}shot"
    if os.path.exists(f"{out}/test.metrics.json"):
        print("done already:", out)
        return
    bs = BS_ZERO if shots == 0 else BS_FEW
    lim = f"--limit {limit}" if limit else ""
    get_ipython().system(
        f'python -m evaluation.generate --model {model} --few_shot {shots} --batch_size {bs} {lim} --out "{out}"')

def show(root, n=2):
    """Print the first n outputs of every run under root."""
    for f in sorted(glob.glob(f"{root}/**/test.generations.jsonl", recursive=True)):
        print("=" * 100, "\n", f.replace(root + "/", ""))
        for line in open(f).readlines()[:n]:
            g = json.loads(line)
            s = g["scores"][0]
            print(f"--- gold {g['answer']} | strict {s['strict_answer']} | lenient {s['lenient_answer']} | {s['gen_tokens']} tok")
            print(g["outputs"][0][:1200])

## 3. Smoke test: 50 questions per run

Takes a few minutes per model. Check the outputs by eye: do chat models end with "The answer is N"? Do base models stop cleanly? Is truncation near 0?

In [ ]:
for model, shots in RUNS:
    run(model, shots, SMOKE, limit=50)
!python -m analysis.summarize "{SMOKE}"

In [ ]:
show(SMOKE)

## 4. Full test set

Roughly 5-15 min per run on L4/A100 and 10-30 min on T4 (to be confirmed by the smoke timings in the `seconds` column).
Free Colab may disconnect; re-run the setup cells and this one, and finished runs are skipped.

In [ ]:
for model, shots in RUNS:
    run(model, shots, RESULTS)

In [ ]:
!python -m analysis.summarize "{RESULTS}"
print("table saved to", f"{RESULTS}/summary.csv")